In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader, WeightedRandomSampler
from transformers import get_cosine_schedule_with_warmup
import timm
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tqdm.auto import tqdm
import sys
sys.path.append(".")

from huggingface_hub import login
from src.dataset import HistologicalImageDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [2]:
CFG = dict(
    data_dir    = "/data/BREAKHIS",  
    img_size    = 224,
    batch_size  = 8,
    num_workers = 4,
    lr_head     = 1e-3,
    lr_backbone = 1e-5,
    epochs      = 20,
    weight_decay= 0.01,
    warmup_steps= 100,
    label_smoothing = 0.1,
    seed        = 42,
    
)

CFG["dataset_name"] = Path(CFG["data_dir"]).name
CFG["output_dir"] = Path(f"checkpoints/{CFG['dataset_name']}/uni_finetuned")

CFG["output_dir"].mkdir(parents=True, exist_ok=True)
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])

In [3]:
train_tf = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation((90, 90))], p=0.5),
    T.RandomApply([T.ColorJitter(0.2, 0.2, 0.1, 0.05)], p=0.5),
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
eval_tf = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

train_ds = HistologicalImageDataset(f"{CFG['data_dir']}/train", transform=train_tf)
val_ds   = HistologicalImageDataset(f"{CFG['data_dir']}/val",   transform=eval_tf)
test_ds  = HistologicalImageDataset(f"{CFG['data_dir']}/test",  transform=eval_tf)

# Sampler bilanciato
counts  = np.bincount(train_ds.labels)
weights = torch.from_numpy((1.0 / counts)[train_ds.labels]).double()
sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

kw = dict(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"],
          pin_memory=True, persistent_workers=True)
train_loader = DataLoader(train_ds, sampler=sampler, drop_last=True, **kw)
val_loader   = DataLoader(val_ds,  shuffle=False, **kw)
test_loader  = DataLoader(test_ds, shuffle=False, **kw)

CLASS_NAMES = train_ds.class_names
N_CLASSES   = len(CLASS_NAMES)
print(f"Classi: {CLASS_NAMES}")

Loading from /data/BREAKHIS/train...


Loading dataset from disk:   0%|          | 0/29 [00:00<?, ?it/s]

Loaded 25880 samples, 8 classes
Class distribution:
  adenosis: 1139 (4.4%)
  fibroadenoma: 3685 (14.2%)
  phyllodes_tumor: 1419 (5.5%)
  tubular_adenoma: 1642 (6.3%)
  ductal_carcinoma: 11717 (45.3%)
  lobular_carcinoma: 1927 (7.4%)
  mucinous_carcinoma: 2446 (9.5%)
  papillary_carcinoma: 1905 (7.4%)
Loading from /data/BREAKHIS/val...
Loaded 6832 samples, 8 classes
Class distribution:
  adenosis: 541 (7.9%)
  fibroadenoma: 692 (10.1%)
  phyllodes_tumor: 423 (6.2%)
  tubular_adenoma: 601 (8.8%)
  ductal_carcinoma: 2769 (40.5%)
  lobular_carcinoma: 601 (8.8%)
  mucinous_carcinoma: 757 (11.1%)
  papillary_carcinoma: 448 (6.6%)
Loading from /data/BREAKHIS/test...
Loaded 6833 samples, 8 classes
Class distribution:
  adenosis: 540 (7.9%)
  fibroadenoma: 693 (10.1%)
  phyllodes_tumor: 423 (6.2%)
  tubular_adenoma: 602 (8.8%)
  ductal_carcinoma: 2769 (40.5%)
  lobular_carcinoma: 602 (8.8%)
  mucinous_carcinoma: 757 (11.1%)
  papillary_carcinoma: 447 (6.5%)
Classi: ['adenosis', 'fibroadenoma',

In [4]:
import torch
import torch.nn as nn
import timm
from peft import LoraConfig
from peft.tuners.lora import LoraModel

class UNILoRAClassifier(nn.Module):
    def __init__(self, n_classes, dropout=0.1):
        super().__init__()

        # Backbone timm puro
        backbone = timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, dynamic_img_size=True)

        # Config LoRA (NO task_type)
        lora_config = LoraConfig(
            r=8,
            lora_alpha=32,
            target_modules=["qkv", "proj", "fc1", "fc2"],  # nomi reali timm
            lora_dropout=0.1,
            bias="none"
        )

        # Inject LoRA direttamente
        self.backbone = LoraModel(backbone, lora_config, adapter_name="default")


        embed_dim = 1024  # ViT-L

        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, n_classes),
        )

    def forward(self, x):
        features = self.backbone(x)   # timm forward puro
        return self.head(features)


model = UNILoRAClassifier(N_CLASSES).to(device)

In [5]:
optimizer = torch.optim.AdamW([
    {"params": [p for p in model.backbone.parameters() if p.requires_grad], 
     "lr": CFG["lr_backbone"]},
    {"params": model.head.parameters(), 
     "lr": CFG["lr_head"]},
], weight_decay=CFG["weight_decay"])

total_steps = CFG["epochs"] * len(train_loader)
scheduler   = get_cosine_schedule_with_warmup(
    optimizer, CFG["warmup_steps"], total_steps)

criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

In [ ]:
from torch.cuda.amp import GradScaler, autocast

scaler = GradScaler()

def run_epoch(model, loader, criterion, optimizer=None, scheduler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0., 0, 0

    with torch.set_grad_enabled(training):
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)

            with autocast():
                logits = model(imgs)
                loss   = criterion(logits, labels)

            if training:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

            total_loss += loss.item() * len(labels)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += len(labels)

    return total_loss / total, correct / total


history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc, best_epoch = 0., 0

for epoch in range(1, CFG["epochs"] + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, scheduler)
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)

    print(f"Epoch {epoch:02d}/{CFG['epochs']}  "
          f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
          f"val_loss={vl_loss:.4f}  val_acc={vl_acc:.4f}")

    if vl_acc > best_val_acc:
        best_val_acc, best_epoch = vl_acc, epoch
        torch.save(model.state_dict(),
                   CFG["output_dir"] / "best_model.pt")
        print(f"  ✅ Salvato best model (val_acc={best_val_acc:.4f})")

print(f"\nBest val_acc={best_val_acc:.4f} @ epoch {best_epoch}")

/tmp/ipykernel_593461/4097670193.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


  0%|          | 0/3235 [00:00<?, ?it/s]

/tmp/ipykernel_593461/4097670193.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["val_loss"],   label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["train_acc"], label="train")
axes[1].plot(epochs, history["val_acc"],   label="val")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("UNI Fine-tuning", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(torch.load(CFG["output_dir"] / "best_model.pt", map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader):
        preds = model(imgs.to(device)).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(all_labels, all_preds, normalize="true")

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix (normalizzata)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()